# Human Action Recognition: Finding a Good Fit with CLIP ViT-B/16

**Goal:** a training curve with **good fit**: training and validation accuracy both rise, flatten, and stay close together
(a small gap), with validation accuracy as high as possible. This is neither underfitting nor overfitting.

**What is compared.** Five variants of **the same CLIP ViT-B/16 model**, each changing one thing from variant A, the best run so far:

| Variant | Change from A | What it tests |
|---|---|---|
| **A** | (baseline) first 6 blocks frozen, drop path 0.1 | The best run so far (gap 4.8 points, smoothest curve) |
| **B** | freeze 8 blocks | Does freezing more layers reduce the gap? |
| **C** | freeze 10 blocks | Is freezing even more too much to learn from? |
| **D** | drop path 0.2 | Does stronger random block-skipping reduce the gap? |
| **E** | drop path 0 | Does drop path matter at all? |

Dropout stays at **0.05** (your teacher's value) in all five.

**How the best variant is chosen (in code):** keep the variants whose **train–validation gap is 5 points or less** at the
best epoch, then take the highest **validation** accuracy. The **test set is only reported**, never used to choose.

> **On Kaggle:** set **Accelerator → GPU T4 x2** and **Internet → On**, attach the *Human Action Recognition (HAR) Dataset*,
> then **Save Version → Save & Run All**. Five runs take about **2.5–3.5 hours**. Finished runs are saved, so a restart resumes.

# Part 0 — Settings
Settings shared by all five variants. The freezing and drop path differ per variant, in Part 3.

In [ ]:
EPOCHS = 25            # maximum epochs; early stopping usually ends sooner
PATIENCE = 3           # stop after 3 epochs without a better validation accuracy
WEIGHT_DECAY = 0.05    # AdamW weight decay
DROPOUT = 0.05         # dropout before the classifier
EMA_DECAY = 0.998      # averages the weights over the last ~500 steps: smooths the curves

# Part 1 — Data

## 1.1 Import libraries
This notebook uses **PyTorch** (a deep-learning framework, like Keras/TensorFlow) and **timm**
("PyTorch Image Models"), a library with hundreds of pretrained image networks, including CLIP.

In [ ]:
import os, gc, glob, copy, math, time, random, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", message="Detected call of")   # harmless scheduler notice with mixed precision

SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Device: ", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU - turn on the GPU accelerator in Settings!")

## 1.2 Find the dataset

In [ ]:
csv_matches = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)
assert csv_matches, "Dataset not found - click 'Add Input' and attach the Human Action Recognition dataset."
DATA_DIR = os.path.dirname(csv_matches[0])
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
OUTPUT_DIR = "/kaggle/working"
print("Data folder:", DATA_DIR)

## 1.3 Load the labels
Each action gets a number (`calling → 0`, `clapping → 1`, …), because networks work with numbers.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "Training_set.csv"))
class_names = sorted(df["label"].unique())
label_to_id = {name: i for i, name in enumerate(class_names)}
df["label_id"] = df["label"].map(label_to_id)
NUM_CLASSES = len(class_names)

print(f"{len(df)} labeled images, {NUM_CLASSES} actions")
print(df["label"].value_counts().sort_index().to_string())

## 1.4 Example images

In [ ]:
sample = df.sample(8, random_state=1)
plt.figure(figsize=(14, 6))
for i, (_, row) in enumerate(sample.iterrows()):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Image.open(os.path.join(TRAIN_IMG_DIR, row["filename"])))
    plt.title(row["label"])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 1.5 Find near-duplicate photos (prevent data leakage)
Some photos in the dataset are near-copies of each other (the same picture saved twice, resized or recompressed).
If one copy lands in the **training** set and the other in the **test** set, the model can simply *remember*
the photo, and the test accuracy looks better than it really is. This is called **data leakage**.

To prevent it we give every photo a **fingerprint** with *difference hashing (dHash)*:
1. Shrink the photo to 9 × 8 pixels in grayscale.
2. For each pixel, record whether it is brighter than its right-hand neighbour → **64 yes/no bits**.

Photos whose fingerprints differ in **4 bits or fewer** are put in the same **group**, and a group is never
split across train, validation and test.

The fingerprint is deliberately **cautious**. Besides real copies, it also matches some *different* photos that
share a large plain background (white studio shots, black borders), because plain areas give identical bits.
Keeping two different photos on the same side of the split doesn't leak anything, so that's harmless.
To see how many matches are real copies, we also compare small 32 × 32 colour thumbnails pixel by pixel.

In [ ]:
def dhash(path):
    img = Image.open(path).convert("L").resize((9, 8), Image.BILINEAR)
    px = np.asarray(img, dtype=np.int16)
    return (px[:, 1:] > px[:, :-1]).flatten()          # 8 rows x 8 comparisons = 64 bits

paths = [os.path.join(TRAIN_IMG_DIR, f) for f in df["filename"]]
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    bits = np.array(list(pool.map(dhash, paths)), dtype=np.float32)   # shape: (images, 64)

# Hamming distance = number of bits that differ between two fingerprints
pair_a, pair_b = [], []
for start in range(0, len(bits), 1024):
    chunk = bits[start:start + 1024]
    agree = chunk @ bits.T + (1 - chunk) @ (1 - bits).T
    a, b = np.nonzero(64 - agree <= 4)
    a = a + start
    keep = b > a                                        # count each pair once, skip self-matches
    pair_a.append(a[keep])
    pair_b.append(b[keep])
pair_a, pair_b = np.concatenate(pair_a), np.concatenate(pair_b)

n = len(df)
_, groups = connected_components(coo_matrix((np.ones(len(pair_a)), (pair_a, pair_b)), shape=(n, n)), directed=False)
df["group"] = groups

# How alike are the matched pairs really? Average pixel difference of 32 x 32 colour thumbnails (0-255 scale)
def thumbnail(path):
    return np.asarray(Image.open(path).convert("RGB").resize((32, 32), Image.BILINEAR), dtype=np.float32)
pixel_diff = np.array([np.abs(thumbnail(paths[a]) - thumbnail(paths[b])).mean() for a, b in zip(pair_a, pair_b)])
is_copy = pixel_diff < 10
same_label = df["label"].values[pair_a] == df["label"].values[pair_b]

print(f"Photo pairs with matching fingerprints: {len(pair_a)}")
if len(pair_a):
    print(f"  clearly copies (thumbnails differ by less than 10 out of 255): {is_copy.sum()}"
          f" - same label in {same_label[is_copy].mean() if is_copy.any() else 0:.0%} of them")
    print(f"  other matches (near-copies such as crops, and look-alikes):      {(~is_copy).sum()}")
print(f"Groups: {groups.max() + 1} groups for {n} images")

**Examples.** The first three pairs are copies of the same photo; the differences are watermarks or image quality.
The last pair is a *look-alike*: two different photos that the fingerprint matched because of their plain backgrounds.

In [ ]:
copies = np.where(is_copy)[0]
copies = copies[np.argsort(-pixel_diff[copies])][:3]      # copies with small visible differences
lookalikes = np.where(~is_copy)[0]
lookalikes = lookalikes[np.argsort(pixel_diff[lookalikes])][len(lookalikes) // 2:][:1]   # a typical look-alike
rows = [(k, "copy") for k in copies] + [(k, "look-alike") for k in lookalikes]
if rows:
    plt.figure(figsize=(8, 3.2 * len(rows)))
    for r, (k, kind) in enumerate(rows):
        for j, idx in enumerate((pair_a[k], pair_b[k])):
            plt.subplot(len(rows), 2, 2 * r + j + 1)
            plt.imshow(Image.open(paths[idx]))
            title = f"{df['filename'][idx]} · {df['label'][idx]}"
            if j == 0:
                title = f"{kind.upper()} (pixel difference {pixel_diff[k]:.1f})\n" + title
            plt.title(title, fontsize=9)
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 1.6 Split into training, validation and test sets
- **Training (80%)** – the model learns from these photos.
- **Validation (10%)** – checked after every epoch to pick the best version of the model.
- **Test (10%)** – used **only once at the very end** to report the final accuracy.

Because the validation set is used to *choose* the best epoch, its score is slightly optimistic.
The untouched **test set** gives an honest final number. *Stratified* keeps the same mix of actions in every set;
*group* keeps near-duplicates together.

(The dataset's own `test` folder has no labels, so it can't measure accuracy. Part 6 explains what we use it for.)

In [ ]:
y_all = df["label_id"].values
all_idx = np.arange(len(df))

split_test = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(split_test.split(all_idx, y_all, groups))

split_val = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=SEED)
tr, va = next(split_val.split(trainval_idx, y_all[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr], trainval_idx[va]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

assert not (set(train_df.group) & set(val_df.group) or set(train_df.group) & set(test_df.group) or set(val_df.group) & set(test_df.group))
print(f"Training:   {len(train_df):>5} images")
print(f"Validation: {len(val_df):>5} images")
print(f"Test:       {len(test_df):>5} images")
print("No near-duplicate group is shared between the sets.")

# Part 2 — Preprocessing

## 2.1 Resize, normalize and augment
**Every image (validation and test):**
- **Resize** to 224 × 224 pixels — the input size CLIP ViT-B/16 was trained on.
- **Normalize** each colour channel with CLIP's own mean and standard deviation, so the inputs look like
  what the network saw during pretraining.

**Training images only — data augmentation** (a new random version every epoch):
- **Random resized crop** — zoom into 40–100% of the photo, so the model sees people at different sizes and positions.
- **Horizontal flip** — a mirrored person is doing the same action.
- **Small rotation** (±10°) and **brightness / contrast** changes (±20%).

Augmentation makes the training set look bigger and more varied, which reduces **overfitting**.

In [ ]:
IMG_SIZE = 224
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10, interpolation=InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

def to_display(t):
    # Undo the normalization so a tensor can be shown as a picture
    mean, std = torch.tensor(CLIP_MEAN).view(3, 1, 1), torch.tensor(CLIP_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

example = Image.open(os.path.join(TRAIN_IMG_DIR, train_df["filename"][0])).convert("RGB")
plt.figure(figsize=(14, 3.5))
plt.subplot(1, 6, 1); plt.imshow(to_display(eval_tf(example))); plt.title("original"); plt.axis("off")
for i in range(5):
    plt.subplot(1, 6, i + 2); plt.imshow(to_display(train_tf(example))); plt.title(f"augmented {i + 1}"); plt.axis("off")
plt.suptitle(f"Action: {train_df['label'][0]}")
plt.tight_layout()
plt.show()

## 2.2 Data loaders
A **Dataset** says how to load one image and its label; a **DataLoader** groups images into **batches of 32**
and loads them in parallel on the CPU while the GPU trains.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() or 2

class HARDataset(Dataset):
    def __init__(self, frame, img_dir, transform, labeled=True):
        self.files = frame["filename"].tolist()
        self.labels = frame["label_id"].tolist() if labeled else [-1] * len(frame)
        self.img_dir, self.transform = img_dir, transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.img_dir, self.files[i])).convert("RGB")
        return self.transform(img), self.labels[i]

def make_loader(frame, img_dir, transform, train, labeled=True):
    return DataLoader(HARDataset(frame, img_dir, transform, labeled),
                      batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)

train_loader = make_loader(train_df, TRAIN_IMG_DIR, train_tf, train=True)
val_loader = make_loader(val_df, TRAIN_IMG_DIR, eval_tf, train=False)
test_loader = make_loader(test_df, TRAIN_IMG_DIR, eval_tf, train=False)

x, y = next(iter(train_loader))
print("Batch of images:", tuple(x.shape), "-> 32 images, 3 colours, 224 x 224 pixels")
print("Batch of labels:", tuple(y.shape))
print("Training steps per epoch:", len(train_loader))

# Part 3 — Training Code (the same for all four models)
- **Layer-wise learning rates for transformers:** the lower layers change slowly and the top adapts more.
  CNNs use one rate with a 5× higher rate for the new classifier.
- **Frozen blocks:** the first `freeze_blocks` blocks (and the embeddings) are not trained.
- **AdamW**, 1 warm-up epoch, then cosine decay; label smoothing 0.1; gradient clipping 1.0; mixed precision.
- **EMA weights:** the smoothed copy is measured, saved and tested.
- **Measured the same way every epoch:** a 2,000-photo sample of training, the validation set and the test set, all on clean photos.
- **Early stopping** on validation accuracy, with patience 3.
- **Each model's own normalization** (its pretraining mean and standard deviation), with the same augmentation as notebook 03.

In [ ]:
import json
from torch.utils.data import Dataset, DataLoader
LAYER_DECAY, HEAD_LR_MULT, LABEL_SMOOTHING, GRAD_CLIP, WARMUP_EPOCHS = 0.75, 5.0, 0.1, 1.0, 1
TRAIN_EVAL_N = 2000
use_amp = device.type == "cuda"
PRETRAINED = True
NUM_WORKERS = os.cpu_count() or 2

# Five CLIP variants: timm name, base learning rate, how many lower blocks are frozen, and drop path.
MODELS = {
    "A: freeze 6, drop path 0.1": dict(name="vit_base_patch16_clip_224.openai", base_lr=5e-5, freeze_blocks=6, drop_path=0.1),
    "B: freeze 8, drop path 0.1": dict(name="vit_base_patch16_clip_224.openai", base_lr=5e-5, freeze_blocks=8, drop_path=0.1),
    "C: freeze 10, drop path 0.1": dict(name="vit_base_patch16_clip_224.openai", base_lr=5e-5, freeze_blocks=10, drop_path=0.1),
    "D: freeze 6, drop path 0.2": dict(name="vit_base_patch16_clip_224.openai", base_lr=5e-5, freeze_blocks=6, drop_path=0.2),
    "E: freeze 6, drop path 0": dict(name="vit_base_patch16_clip_224.openai", base_lr=5e-5, freeze_blocks=6, drop_path=0.0),
}

class Photos(Dataset):
    def __init__(self, frame, tf):
        self.files, self.labels, self.tf = frame["filename"].tolist(), frame["label_id"].tolist(), tf
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        return self.tf(Image.open(os.path.join(TRAIN_IMG_DIR, self.files[i])).convert("RGB")), self.labels[i]

def make_transforms(mean, std):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(10, interpolation=InterpolationMode.BILINEAR),
        transforms.ColorJitter(0.2, 0.2), transforms.ToTensor(), transforms.Normalize(mean, std)])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    return train_tf, eval_tf

def dl(frame, tf, train):
    return DataLoader(Photos(frame, tf), batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=use_amp)

def build_model(name, dropout, drop_path):
    try:
        net = timm.create_model(name, pretrained=PRETRAINED, num_classes=NUM_CLASSES,
                                drop_rate=dropout, drop_path_rate=drop_path)
    except Exception as e:
        raise RuntimeError("Could not download the pretrained weights. In Kaggle, open Settings, turn Internet ON, "
                           "and run the notebook again.") from e
    return net.to(device)

def layer_id(name, num_blocks):
    if name.startswith(("cls_token", "pos_embed", "patch_embed", "norm_pre")):
        return 0
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1
    return num_blocks + 1

def param_groups(net, base_lr, freeze_blocks, weight_decay):
    head_ids = {id(p) for p in net.get_classifier().parameters()}
    is_vit = hasattr(net, "blocks")
    num_blocks = len(net.blocks) if is_vit else 0
    groups = {}
    for name, p in net.named_parameters():
        lid = layer_id(name, num_blocks) if is_vit else 1
        is_head = id(p) in head_ids
        if not is_head and lid <= freeze_blocks:
            p.requires_grad_(False)                       # frozen: not trained
            continue
        if is_head:
            lr = base_lr * HEAD_LR_MULT
        elif is_vit:
            lr = base_lr * LAYER_DECAY ** (num_blocks + 1 - lid)
        else:
            lr = base_lr
        no_decay = p.ndim <= 1 or name in ("cls_token", "pos_embed")
        g = groups.setdefault((lid, is_head, no_decay), {
            "params": [], "lr": lr, "weight_decay": 0.0 if no_decay else weight_decay})
        g["params"].append(p)
    return list(groups.values())

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))
        for ema_v, v in zip(self.model.state_dict().values(), model.state_dict().values()):
            if ema_v.dtype.is_floating_point:
                ema_v.mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                ema_v.copy_(v)

@torch.no_grad()
def predict(net, loader):
    net.eval()
    probs, labels = [], []
    for x, y in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            probs.append(net(x.to(device, non_blocking=True)).float().softmax(1).cpu())
        labels.append(y)
    return torch.cat(probs), torch.cat(labels)

def measure(net, loader):
    p, y = predict(net, loader)
    return F.nll_loss(torch.log(p + 1e-9), y).item(), (p.argmax(1) == y).float().mean().item()

def train_model(label):
    cfg = MODELS[label]
    folder = os.path.join(OUTPUT_DIR, "runs7", "".join(c if c.isalnum() else "_" for c in label))
    if os.path.exists(os.path.join(folder, "result.json")):
        with open(os.path.join(folder, "result.json")) as f:
            r = json.load(f)
        print(f"[{label}] loaded the finished run")
        return pd.read_csv(os.path.join(folder, "history.csv")), r

    seed_everything(SEED)
    t_start = time.time()
    net = build_model(cfg["name"], DROPOUT, cfg["drop_path"])
    mean, std = timm.data.resolve_model_data_config(net)["mean"], timm.data.resolve_model_data_config(net)["std"]
    train_tf, eval_tf = make_transforms(mean, std)
    train_dl = dl(train_df, train_tf, True)
    val_dl = dl(val_df, eval_tf, False)
    test_dl = dl(test_df, eval_tf, False)
    train_eval_dl = dl(train_eval_df, eval_tf, False)
    groups = param_groups(net, cfg["base_lr"], cfg["freeze_blocks"], WEIGHT_DECAY)
    trainable = sum(p.numel() for g in groups for p in g["params"])
    print(f"[{label}] trainable weights: {trainable / 1e6:.1f}M of {sum(p.numel() for p in net.parameters()) / 1e6:.1f}M")
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    steps = len(train_dl)
    total, warm = EPOCHS * steps, WARMUP_EPOCHS * steps
    def lr_factor(s):
        if s < warm:
            return s / warm
        return max(0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))), 0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    ema = EMA(net, EMA_DECAY)
    history, best, best_state, bad, step = [], -1.0, None, 0, 0
    for epoch in range(1, EPOCHS + 1):
        net.train()
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(net(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update(net, step)
            step += 1
        row = {"epoch": epoch}
        for s, d in (("train", train_eval_dl), ("val", val_dl), ("test", test_dl)):
            row[f"{s}_loss"], row[f"{s}_acc"] = measure(ema.model, d)
        history.append(row)
        improved = row["val_acc"] > best
        if improved:
            best, bad = row["val_acc"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in ema.model.state_dict().items()}
        else:
            bad += 1
        print(f"[{label}] epoch {epoch:2d} | train {row['train_acc']:.1%} | val {row['val_acc']:.1%} | "
              f"test {row['test_acc']:.1%}{'  <- best' if improved else ''}")
        if bad >= PATIENCE:
            print(f"[{label}] early stopping: no better validation accuracy for {PATIENCE} epochs")
            break
    ema.model.load_state_dict(best_state)
    h = pd.DataFrame(history)
    b = h.loc[h["val_acc"].idxmax()]
    bumps = h["val_acc"].diff().abs().dropna() * 100
    test_probs, test_labels = predict(ema.model, test_dl)
    result = {"model": label, "trainable (M)": round(trainable / 1e6, 1), "base lr": cfg["base_lr"],
              "frozen blocks": cfg["freeze_blocks"], "drop path": cfg["drop_path"], "epochs run": len(h), "best epoch": int(b["epoch"]),
              "train acc": float(b["train_acc"]), "val acc": float(b["val_acc"]), "test acc": float(b["test_acc"]),
              "train-val gap": float(b["train_acc"] - b["val_acc"]),
              "bumpiness (pts)": round(float(bumps.mean()), 2), "validation drops": int((h["val_acc"].diff() < 0).sum()),
              "minutes": round((time.time() - t_start) / 60, 1)}
    os.makedirs(folder, exist_ok=True)
    h.to_csv(os.path.join(folder, "history.csv"), index=False)
    np.save(os.path.join(folder, "test_probs.npy"), test_probs.numpy())
    np.save(os.path.join(folder, "test_labels.npy"), test_labels.numpy())
    with open(os.path.join(folder, "result.json"), "w") as f:
        json.dump(result, f)
    del opt, sched, scaler, criterion, best_state, ema, net, train_dl, val_dl, test_dl, train_eval_dl
    gc.collect()
    if use_amp:
        torch.cuda.empty_cache()
    print(f"[{label}] best epoch {result['best epoch']}: validation {result['val acc']:.1%}, test {result['test acc']:.1%}")
    return h, result

# Part 4 — Train the Five Variants
Each variant is trained with the same settings and early stopping. This takes about 2.5–3.5 hours on Kaggle's GPU.

In [ ]:
train_eval_df = train_df.sample(min(TRAIN_EVAL_N, len(train_df)), random_state=SEED).reset_index(drop=True)
histories, results = {}, []
for label in MODELS:
    h, r = train_model(label)
    histories[label] = h
    results.append(r)
res = pd.DataFrame(results)

# Part 5 — Results and Curves
## 5.1 Comparison table
- **Train–validation gap** at the best epoch: small (about 5 points or less) means a good fit.
  A large gap means overfitting. Low training accuracy as well as low validation means underfitting.
- **Bumpiness:** average change in validation accuracy between epochs (smaller is smoother).

In [ ]:
display(res.style.format({"train acc": "{:.1%}", "val acc": "{:.1%}", "test acc": "{:.1%}",
                          "train-val gap": "{:+.1%}", "bumpiness (pts)": "{:.2f}"}))

fig, axes = plt.subplots(2, len(MODELS), figsize=(4.6 * len(MODELS), 7.5), squeeze=False)
for j, (label, h) in enumerate(histories.items()):
    best = int(h.loc[h["val_acc"].idxmax(), "epoch"])
    for i, metric in enumerate(("acc", "loss")):
        ax = axes[i, j]
        for s, style, lab in (("train", "o-", "train"), ("val", "s-", "validation"), ("test", "^--", "test")):
            vals = h[f"{s}_{metric}"] * (100 if metric == "acc" else 1)
            ax.plot(h["epoch"], vals, style, ms=3, label=lab)
        ax.axvline(best, color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"{label}\n{metric} (best epoch {best})", fontsize=9)
        ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
        if j == 0 and i == 0:
            ax.legend(fontsize=8)
fig.suptitle("Same measurement for train, validation and test after every epoch")
plt.tight_layout(); plt.show()

## 5.2 Choosing the model
Rule: keep the variants whose train–validation gap is 5 points or less (good fit), then take the highest **validation** accuracy.
The test set is only reported.

In [ ]:
GOOD_FIT_GAP = 0.05
eligible = res[res["train-val gap"] <= GOOD_FIT_GAP]
if len(eligible):
    pick = eligible.sort_values("val acc", ascending=False).iloc[0]
    print(f"RECOMMENDED: {pick['model']}  (validation {pick['val acc']:.1%}, test {pick['test acc']:.1%}, "
          f"gap {pick['train-val gap']:+.1%})")
else:
    pick = res.sort_values("val acc", ascending=False).iloc[0]
    print("No model has a gap of 5 points or less. Closest fit by validation accuracy:")
    print(f"  {pick['model']}  (validation {pick['val acc']:.1%}, gap {pick['train-val gap']:+.1%})")
    print("Read the curves: a large gap means overfitting; both low means underfitting.")

# Part 6 — The Chosen Variant in Detail
Per-action results and the most common mistakes, on the **test set**, for the model chosen in Part 5.

In [ ]:
folder = os.path.join(OUTPUT_DIR, "runs7", "".join(c if c.isalnum() else "_" for c in pick["model"]))
test_probs = np.load(os.path.join(folder, "test_probs.npy"))
y_true = np.load(os.path.join(folder, "test_labels.npy"))
y_pred = test_probs.argmax(1)
print(f"{pick['model']}: test accuracy {(y_pred == y_true).mean():.1%}")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=60, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion matrix: {pick['model']} (test set)")
fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

pairs = sorted(((cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j), reverse=True)[:5]
print("Most common mix-ups:")
for n, i, j in pairs:
    print(f"  {class_names[i]:>20} predicted as {class_names[j]:<20} {n} times")

# Summary
- **Five variants of CLIP ViT-B/16:** the same code, changing only freezing and drop path.
- **Chosen by a written rule:** good fit first (gap of 5 points or less), then validation accuracy. The test set only reports.
- **Curves:** Part 5 shows every variant's accuracy and loss on one grid, so you can compare them side by side.